In [ ]:
## conda install lightgbm -c conda-forge

In [225]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statistics
import csv
import copy
import pickle

from sklearn.metrics import f1_score, accuracy_score, ConfusionMatrixDisplay, classification_report
from sklearn.model_selection import KFold, StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingClassifier
import openml
import lightgbm as lgbm
from lightgbm import LGBMClassifier

from tqdm import tqdm

from itertools import product, cycle
from functools import partial

from sklearn.base import clone as sk_clone

from helpers.persistence import save_var, load_var
from helpers.progress_bar import ProgressBar

from joblib import Parallel, delayed

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

In [226]:
settings = {
    'method': 'bootstrap',
    'kfolds': 5,
}

In [227]:
pbar = False

In [228]:
def set_random_seeds():
    np.random.seed(0)


set_random_seeds()

In [229]:
def find_best_params(params_range, clf, train_df, train_y, valid_df, valid_y):
    best_params = False
    best_val = 0

    param_combinations = list(product(*params_range.values()))

    for i, params in enumerate(param_combinations):
        params = {k: v for k, v in zip(params_range.keys(), params)}

        # clone to get the unfitted yet a true copy of the classifier
        # didn't change the output
        set_random_seeds()
        model = sk_clone(clf)
        model.set_params(**params)

        model.fit(train_df, train_y)

        y_pred = model.predict(valid_df)
        score = f1_score(valid_y, y_pred, average='weighted')

        pbar and pbar.set_description(
            f'Validating: {i+1} / {len(param_combinations)} | val: {score:.3f} (best={best_val:.3f})')

        if score > best_val:
            best_val = score
            best_params = params
    return best_params

In [230]:
callbacks = [
    # lgbm.early_stopping(10, verbose=0),
    lgbm.log_evaluation(period=0)
]


def train_and_test(dataset_name):
    dataset = openml.datasets.get_dataset(dataset_name)

    X, y, categorical_indicator, attribute_names = dataset.get_data(
        dataset_format="array", target=dataset.default_target_attribute
    )
    df = pd.DataFrame(X, columns=attribute_names)
    cat_mask = np.array(categorical_indicator)

    numeric_features = df.columns[~cat_mask]
    categorical_features = df.columns[cat_mask]

    numeric_transformer = Pipeline(
        steps=[
            # ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]
    )

    categorical_transformer = Pipeline(
        steps=[
            ("encoder", OneHotEncoder(handle_unknown="ignore", sparse=False)),
        ]
    )
    preprocessor = ColumnTransformer(
        transformers=[
            # ("num", numeric_transformer, numeric_features),
            ("cat", categorical_transformer, categorical_features),
        ],
        remainder='passthrough'
    )

    params_range = {
        "classifier__n_estimators": [50, 80, 110],
        "classifier__max_depth": [-1, 2, 5, 10, 15]
    }

    clf = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("scaler", StandardScaler()),
            ("classifier", LGBMClassifier(
                n_jobs=-1,  # this is -1 by default
                n_estimators=100, random_state=42)),
        ]
    )

    k_folds = settings['kfolds']
    cv = StratifiedKFold(n_splits=k_folds, random_state=42, shuffle=True)

    pbar and pbar.add_prefix('creating folds')
    fold_limit = 5 if settings['method'] == 'kfold' else 30

    kfold_splits = list(cv.split(df, y))

    scores = []
    y_preds = []
    y_trues = []
    best_params_list = []

    # run k fold in loop
    for fold_counter in range(fold_limit):
        pbar and pbar.edit_last_prefix(
            f'fold={fold_counter+1}/{fold_limit} | ')

        if settings['method'] == 'kfold':
            train, test = kfold_splits[fold_counter]

            # divide the data for train and test
            train_df, train_y = df.iloc[train], y[train]
            test_df, test_y = df.iloc[test], y[train]
        else:
            train_df, test_df, train_y, test_y = train_test_split(df, y,
                                                                  stratify=y,
                                                                  shuffle=True,
                                                                  test_size=1/5,
                                                                  random_state=fold_counter)

        train_df, valid_df, train_y, valid_y = train_test_split(train_df, train_y,
                                                                stratify=train_y,
                                                                test_size=1/8, random_state=42)

        # fit the clf pipeline except classifier
        clf_temp = Pipeline(clf.steps[:-1])
        valid_X = clf_temp.fit_transform(valid_df, valid_y)
        clf_temp.steps.append(clf.steps[-1])

        best_params = False
        best_val = 0
        best_model = False
        best_loss = np.inf

        param_combinations = list(product(*params_range.values()))

        for i, params in enumerate(param_combinations):

            params = {k: v for k, v in zip(params_range.keys(), params)}

            # clone to get the unfitted yet a true copy of the classifier
            # cloneing didn't change the output - So fitting again reset the classifer
            set_random_seeds()
            model = sk_clone(clf_temp)
            model.set_params(**params)

            model.fit(train_df, train_y,
                      classifier__eval_set=[(valid_X, valid_y)],
                      classifier__callbacks=callbacks,
                      )

            val_metrics = model.steps[-1][1]._evals_result['valid_0']
            k, v = next(iter(val_metrics.items()))
            mean_loss = np.mean(v)

            y_pred = model.predict(valid_df)
            score = f1_score(valid_y, y_pred, average='weighted')

            val_metric = mean_loss  # score
            best_metric = best_loss  # best_val
            pbar and pbar.set_description(
                f'Validating: {i+1} / {len(param_combinations)} | val_{k}: {val_metric:.3f} (best={best_metric:.3f})')

            # is_better = score>best_val # use when using validation score
            is_better = mean_loss < best_loss  # lgbm also provides logloss
            if is_better:
                best_val = score
                best_loss = mean_loss
                best_params = params
                best_model = model

        # use the best model that has the highest validation score
        y_pred = best_model.predict(test_df)

        score = f1_score(test_y, y_pred, average='weighted')

        scores.append(score)
        y_preds.append(y_pred)
        y_trues.append(test_y)
        best_params_list.append(best_params)

    results = {
        'scores': scores,
        'predicted': y_preds,
        'actual': y_trues,
        'best_params': best_params_list,
    }

    save_var(results, f'./saved_vars/lgbm_{dataset_name}.pkl')

    return scores

In [231]:
bool({}) == True, bool({'a': 2}) == True

(False, True)

In [232]:
def get_results(id):
    rows = []
    results = load_var(f'./saved_vars/lgbm_{id}.pkl', False) or {}
    results = {}

    scores = train_and_test(id) if not bool(results) else results['scores']

    for i, s in enumerate(scores):
        rows.append([id, 'lgbm', i, s])

    return pd.DataFrame(rows, columns=['id', 'corruption', 'fold', 'test_score'])

In [233]:
id_list = [1049,458,  469,  1050, 1063, 1067, 1068, 1461, 4538,    3,   11,
           23,   31,   37,   44,   46,   50,   54,  151, 1485]

pbar = ProgressBar(id_list)
# df_list = Parallel(n_jobs=5)(delayed(get_results)(id)
#                                      for id in id_list)

df_list = []
for id in pbar:
    pbar.clear_prefix()
    # if id == 4538:
    #     continue
    pbar.add_prefix(f'Doing id={id}')
    r = get_results(id)
    df_list.append(r)


df = pd.concat(df_list)
df.to_csv('./exports/lgbm.csv')

Doing id=50 | fold=30/30 |  Validating: 15 / 15 | val_binary_logloss: 0.121 (best=0.120):  85%|████████▌ | 17/20 [13:42<02:35, 51.82s/it] Encountered unsupported pickle protocol when loading dataset 54 from 'C:\Users\shour\.openml\org\openml\www\datasets\54\dataset.pkl.py3'. Error message was: unsupported pickle protocol: 5. We will continue loading data from the arff-file, but this will be much slower for big datasets. Please manually delete the cache file if you want OpenML-Python to attempt to reconstruct it.
Doing id=1485 | fold=30/30 |  Validating: 15 / 15 | val_binary_logloss: 0.441 (best=0.431): 100%|██████████| 20/20 [18:06<00:00, 54.32s/it]


In [234]:
tmp = df.groupby('id').test_score.agg(['mean', 'std']).reset_index()
df0 = pd.read_csv('./exports/tabular_description.csv')

df0.set_index('id').join(tmp.set_index('id')).dropna()

,name,samples,features,categories,classes,mean,std
id,,,,,,,
3,kr-vs-kp,3196,36,36,2,0.967399,0.025533
11,balance-scale,625,4,0,3,0.854272,0.016645
23,cmc,1473,9,7,3,0.546477,0.029273
31,credit-g,1000,20,13,2,0.722024,0.024063
37,diabetes,768,8,0,2,0.751612,0.032772
44,spambase,4601,57,0,2,0.952534,0.008065
46,splice,3190,61,60,3,0.954621,0.006307
50,tic-tac-toe,958,9,9,2,0.990251,0.006102
54,vehicle,846,18,0,4,0.757722,0.026359


In [235]:
df_gbt = pd.read_csv('./exports/gbt.csv')

In [236]:
pd.concat([df, df_gbt]).pivot_table(index='id', columns='corruption', values='test_score', 
                                    aggfunc=lambda x: f'{x.mean():.3f} ({x.std():.3f})')

corruption,gbt,lgbm
id,,
3,0.994 (0.004),0.967 (0.026)
11,0.849 (0.019),0.854 (0.017)
23,0.537 (0.032),0.546 (0.029)
31,0.732 (0.031),0.722 (0.024)
37,0.752 (0.028),0.752 (0.033)
44,0.948 (0.006),0.953 (0.008)
46,0.959 (0.009),0.955 (0.006)
50,0.983 (0.012),0.990 (0.006)
54,0.738 (0.029),0.758 (0.026)
